## Configure API KEYS

In [1]:
import os
from getpass import getpass

try:
    from dotenv import load_dotenv
    load_dotenv()
except Exception:
    pass
if not os.getenv("GOOGLE_API_KEY"):
    os.environ["GOOGLE_API_KEY"] = getpass("Enter GOOGLE_API_KEY: ")

if not os.getenv("GROQ_API_KEY"):
    os.environ["GROQ_API_KEY"] = getpass("Enter GROQ_API_KEY: ")

if not os.getenv("TAVILY_API_KEY"):
    os.environ["TAVILY_API_KEY"] = getpass("Enter TAVILY_API_KEY: ")

if not os.getenv("PINECONE_API_KEY"):
    os.environ["PINECONE_API_KEY"] = getpass("Enter PINECONE_API_KEY: ")

print("GOOGLE_API_KEY configured:", bool(os.getenv("GOOGLE_API_KEY")))
print("GROQ_API_KEY configured:", bool(os.getenv("GROQ_API_KEY")))
print("TAVILY_API_KEY configured:", bool(os.getenv("TAVILY_API_KEY")))
print("PINECONE_API_KEY configured:", bool(os.getenv("PINECONE_API_KEY")))

# print("GOOGLE_API_KEY configured:", os.getenv("GOOGLE_API_KEY"))
# print("GROQ_API_KEY configured:", os.getenv("GROQ_API_KEY"))
# print("TAVILY_API_KEY configured:", os.getenv("TAVILY_API_KEY"))
# print("PINECONE_API_KEY configured:", os.getenv("PINECONE_API_KEY"))

GOOGLE_API_KEY configured: True
GROQ_API_KEY configured: True
TAVILY_API_KEY configured: True
PINECONE_API_KEY configured: True


### 1. Load the Document 

In [2]:
from langchain_community.document_loaders import WebBaseLoader

source_url = "https://docs.langchain.com/oss/python/langgraph/agentic-rag"

loader = WebBaseLoader(
    web_path=source_url,
    requests_kwargs = {
        "headers": {
            "User-Agent": "Mozilla/5.0 Agentic-RAG-Industry-Demo"
        }
    }
)
raw_docs = loader.load()

print(f"Loaded document: {len(raw_docs)}\n documents from: {source_url}")
print("Source: ", raw_docs[0].metadata.get("source"))
print("\nPreview:\n")
print(raw_docs[0].page_content[:1000])

C:\Users\PRARINAM DERIA\AppData\Local\Temp\ipykernel_28428\3264026688.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import WebBaseLoader
USER_AGENT environment variable not set, consider setting it to identify your requests.


Loaded document: 1
 documents from: https://docs.langchain.com/oss/python/langgraph/agentic-rag
Source:  https://docs.langchain.com/oss/python/langgraph/agentic-rag

Preview:







Build a custom RAG agent with LangGraph - Docs by LangChain









Documentation IndexFetch the complete documentation index at: /llms.txtUse this file to discover all available pages before exploring further.Skip to main contentWould you notice if your agent got worse in production? Tell us in our annual survey for the chance to win LangChain swag.Docs by LangChain home pageBuildSearch...⌘KAsk AIGitHubTry LangSmithTry LangSmithSearch...NavigationLangGraphBuild a custom RAG agent with LangGraphOverviewManaged Deep AgentsDeep AgentsLangChainLangGraphOpenWikiIntegrationsLearnReferenceContributePythonLearnTutorialsDeep AgentsData AnalysisDeep ResearchContent BuilderRAGLangChainDeep AgentSemantic searchSQL agentVoice agentMulti-agentSubagents: Personal assistantHandoffs: Customer supportRouter: Knowledge base

### 2. Split the document into Chunk

In [3]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size = 1000,
    chunk_overlap = 100,
    add_start_index = True,
)

chunks = splitter.split_documents(raw_docs)

print("Total chunks:", len(chunks))
print("\nFirst chunk preview:\n")
print(chunks[0].page_content[:900])

Total chunks: 28

First chunk preview:

Build a custom RAG agent with LangGraph - Docs by LangChain


### Create Embeddings Using Gemini Model

In [4]:
from google import genai
from google.genai import types
# from langchain_google_genai import GoogleGenerativeAIEmbeddings

gemini_client = genai.Client(api_key=os.getenv("GOOGLE_API_KEY"))

embeddings = gemini_client.models.embed_content(
    model="gemini-embedding-2",
    contents="What is agentic RAG in LangChain and how does it work?",
    config = types.EmbedContentConfig(
        output_dimensionality=768
    )
)


print("Gemini Embedding Result: ", embeddings)

print("Gemini Embedding Result: ", embeddings.embeddings)
[embedding_obj] = embeddings.embeddings
embedding_length = len(embedding_obj.values)
print("Gemini Embedding Result Length: ", embedding_length)

Gemini Embedding Result:  sdk_http_response=HttpResponse(
  headers=<dict len=11>
) embeddings=[ContentEmbedding(
  values=[
    0.01283944,
    0.013940322,
    0.02108872,
    0.027043901,
    -0.007166158,
    <... 763 more items ...>,
  ]
)] metadata=None
Gemini Embedding Result:  [ContentEmbedding(
  values=[
    0.01283944,
    0.013940322,
    0.02108872,
    0.027043901,
    -0.007166158,
    <... 763 more items ...>,
  ]
)]
Gemini Embedding Result Length:  768


### 3. Custom Gemini Embeddings Class

In [5]:
import os
from google import genai
from google.genai import types
from langchain_core.embeddings import Embeddings


# Custom gemini embeddings object
class GeminiSDKEmbeddings(Embeddings):

  def __init__(self, model: str = "gemini-embedding-2", output_dim: int = 768):
    self.client = genai.Client(api_key=os.getenv("GOOGLE_API_KEY"))
    self.model = model
    self.config = types.EmbedContentConfig(output_dimensionality=output_dim)

  def embed_documents(self, texts: list[str]) -> list[list[float]]:
    response = self.client.models.embed_content(
        model=self.model, contents=texts, config=self.config
    )
    return [e.values for e in response.embeddings]

  def embed_query(self, text: str) -> list[float]:
    response = self.client.models.embed_content(
        model=self.model, contents=text, config=self.config
    )
    return response.embeddings[0].values


# Instantiate wrapper
gemini_embeddings = GeminiSDKEmbeddings(
    model="gemini-embedding-2", output_dim=768
)


### 4. Create the Pinecone Vector Database

In [6]:
from pinecone import Pinecone, ServerlessSpec
from langchain_pinecone import PineconeVectorStore
import time

INDEX_NAME = "uniquery-agentic-rag-kb"
NAMESPACE = "uniquery-langgraph-agentic-rag"

pc = Pinecone(api_key=os.getenv("PINECONE_API_KEY"))

existing_indexes = [index["name"] for index in pc.list_indexes()]

if INDEX_NAME not in existing_indexes:
    pc.create_index(
        name=INDEX_NAME,
        dimension=768,
        metric="cosine",
        spec=ServerlessSpec(
            cloud="aws",
            region="us-east-1"
        ),
    )

    # wait until pinecone reports new index is ready
    while not pc.describe_index(INDEX_NAME).status["ready"]:
        print("Waiting for index to be ready...")
        time.sleep(5)



print("Pinecone index ready:", INDEX_NAME)

# Upload the document chunks and create the LangChain vector store.
vectorstore = PineconeVectorStore.from_documents(
    documents=chunks,
    embedding=gemini_embeddings,
    index_name=INDEX_NAME,
    namespace=NAMESPACE,
)

retriever = vectorstore.as_retriever(
    search_kwargs={
        "k": 4,
        "namespace": NAMESPACE
    }
)
print("Vector store created and retriever initialized.")
print("Retriever search_kwargs:", retriever)

d:\GenAi\UniQuery\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Pinecone index ready: uniquery-agentic-rag-kb
Vector store created and retriever initialized.
Retriever search_kwargs: tags=['PineconeVectorStore', 'GeminiSDKEmbeddings'] vectorstore=<langchain_pinecone.vectorstores.PineconeVectorStore object at 0x000001AD9A41E900> search_kwargs={'k': 4, 'namespace': 'uniquery-langgraph-agentic-rag'}


In [7]:
## Load Existing Index

from pinecone import Pinecone
from langchain_pinecone import PineconeVectorStore

INDEX_NAME = "uniquery-agentic-rag-kb"
NAMESPACE = "uniquery-langgraph-agentic-rag"

pc = Pinecone(api_key = os.getenv("PINECONE_API_KEY"))

# Load Index
index = pc.Index(INDEX_NAME)

# Connect Existing index with lagchain
vectorstore = PineconeVectorStore(
    index=index,
    embedding=gemini_embeddings,
    namespace=NAMESPACE
)

retriever = vectorstore.as_retriever(
    search_kwargs={
        "k": 4,
        "namespace": NAMESPACE
    }
)
print("Pinecone retriever loaded:", retriever)
print("Existing Pinecone index loaded and retriever initialized.")

Pinecone retriever loaded: tags=['PineconeVectorStore', 'GeminiSDKEmbeddings'] vectorstore=<langchain_pinecone.vectorstores.PineconeVectorStore object at 0x000001AD9A4FF390> search_kwargs={'k': 4, 'namespace': 'uniquery-langgraph-agentic-rag'}
Existing Pinecone index loaded and retriever initialized.


### Test Private KB Retrieval

In [8]:
test_question = "What happens if retrieved documents are not relevant in Agentic RAG?"

kb_docs = retriever.invoke(test_question)

for i, docs in enumerate(kb_docs):
    print(f"\nRetrieved Document {i+1}:")
    print("Source:", docs.metadata.get("source"))
    print("Content Preview:", docs.page_content[:500], "...\n")


Retrieved Document 1:
Source: https://docs.langchain.com/oss/python/langgraph/agentic-rag
Content Preview: Build a custom RAG agent with LangGraph - Docs by LangChain ...


Retrieved Document 2:
Source: https://docs.langchain.com/oss/python/langgraph/agentic-rag
Content Preview: Build a custom RAG agent with LangGraph - Docs by LangChainDocumentation IndexFetch the complete documentation index at: /llms.txtUse this file to discover all available pages before exploring further.Skip to main contentDocs by LangChain home pageBuildSearch...⌘KAsk AIGitHubTry LangSmithTry LangSmithSearch...NavigationLangGraphBuild a custom RAG agent with LangGraphOverviewDeep AgentsManaged Deep AgentsLangChainLangGraphOpenWikiIntegrationsLearnReferenceContributePythonLearnTutorialsDeep Agents ...


Retrieved Document 3:
Source: https://docs.langchain.com/oss/python/langgraph/agentic-rag
Content Preview: Build a custom RAG agent with LangGraph - Docs by LangChainDocumentation IndexFetch the complete document

## IMPORT GEMINI LLM

In [10]:
from google import genai
import json

client = genai.Client()

llm = client.interactions.create(
    model="gemini-3.8-flash",
    input="Explain how Agentic AI works in 2 lines."
)
print("llm created with model:", llm, type(llm))
print("Answer: ", llm.output_text)

llm created with model: status='completed' agent=None agent_config=None cached_content=None continuation_token=None created='2026-11-15T13:19:25Z' environment=None environment_id=None errors=None generation_config=None id='v1_ChctYV81YXRfR0F2R042OTRQODVLY29RYxIXLWFfNWF0X0dBdkdONjk0UDg1S2NvUWM' input=None labels=None model='gemini-3.8-flash' output_audio=None output_image=None output_text='Agentic AI works by breaking high-level goals into multi-step plans and autonomously executing them using external tools and data.\nIt continuously evaluates its own progress, learns from feedback, and self-corrects its actions in a loop until the objective is achieved.' output_video=None previous_interaction_id=None response_format=None response_mime_type=None response_modalities=None safety_settings=None service_tier='standard' steps=[ThoughtStep(signature='ErQOCrEOAWkUfROkVA4zPmNTbQWMGE+v3OwAUQ7GU87lmdcQNlioycoiKRY/pzKlG5JSqTdugGpkNChaKdcdZkK1fynTjjC2NLHtercysmizfvKt5jqMWZ7xm31N/VNr9SgRkPt/PQOKnj7l

### 5. Initialized Tavily web search

In [12]:
from langchain_tavily import TavilySearch
web_search = TavilySearch(
    max_results=5,
    topic = "general",
    include_answer = True,
    include_raw_content = True,
    # api_key=os.getenv("TAVILY_API_KEY"),
)
print("Tavily search initialized: ", web_search)


Tavily search initialized:  max_results=5 topic='general' include_answer=True include_raw_content=True api_wrapper=TavilySearchAPIWrapper(tavily_api_key=SecretStr('**********'), api_base_url=None)


### 6. Define Structured Decisions
###### We need reliable yes/no or route outputs from the LLM.

###### Structured outputs help avoid parsing messy natural language.

In [13]:
from typing import List, Optional, Literal
from typing_extensions import TypedDict
from pydantic import BaseModel, Field
from langchain_core.documents import Document

class RouteDecision(BaseModel):
    route: Literal["KB", "Direct"] = Field(description = "use kb for needing Agentic Rag docs, direct for greetings/sample chat")


class EvidenceGrade(BaseModel):
    grade: Literal["Good", "Weak"] = Field(description = "Good means evidence can answer question, Weak means evidence is not sufficient to answer question")


class AgentState(TypedDict):
    question: str
    current_query: str
    kb_docs: List[Document]
    web_results: str
    kb_grade: str
    answer: str
    source_used: str
    retry_count: int

#### 7. Node 1 -- Route the Question

###### The Router decides:
###### kb: use private knowledge base
###### direct: answer without retrieval

###### In a real company assistant, this step may also route between many sources: HR docs, product docs, SQL Databases, web search, etc.

In [14]:
# 3. Request structured JSON using client.interactions.create
# interaction = client.interactions.create(
#     model="gemini-3.8-flash",
#     input="I want to cancel my subscription and get a refund.",
#     response_format={
#         "type": "text",
#         "mime_type": "application/json",
#         "schema": RouteDecision.model_json_schema()  # Generates the schema for Gemini
#     }
# )




import json
from typing import Literal, TypedDict, Type, TypeVar
from google import genai
from pydantic import BaseModel, Field

# ==========================================
# 1. DEFINE YOUR STATE & SCHEMAS
# ==========================================

# Mock LangGraph State structure
# class AgentState(TypedDict):
#     question: str
#     current_query: str
#     source_used: Literal["kb", "direct"]


# # Pydantic schema enforcing Gemini's output
# class RouteDecision(BaseModel):
#     route: Literal["kb", "direct"] = Field(
#         description="Route to 'kb' for RAG/technical topics, or 'direct' for general chit-chat."
#     )


# ==========================================
# 2. THE GEMINI NATIVE WRAPPER CLASS
# ==========================================
T = TypeVar('T', bound=BaseModel)

class StructuredGeminiRouter:
    def __init__(self, model_name: str = "gemini-3.8-flash"):
        self.client = genai.Client()
        self.model_name = model_name

    def with_structured_output(self, schema: Type[T]) -> callable:
        def invoke_llm(prompt: str) -> T:
            # Replicates the structured JSON mode natively
            interaction = self.client.interactions.create(
                model=self.model_name,
                input=prompt,
                response_format={
                    "type": "text",
                    "mime_type": "application/json",
                    "schema": schema.model_json_schema()
                }
            )
            return schema.model_validate_json(interaction.output_text)
        return invoke_llm


# Initialize the Gemini router method
gemini_factory = StructuredGeminiRouter(model_name="gemini-3.8-flash")
router_llm = gemini_factory.with_structured_output(RouteDecision)


# ==========================================
# 3. REPLICATED ROUTER NODES & CONDITIONAL ROUTING
# ==========================================

def route_question(state: AgentState):
    question = state["question"]

    # Invoking the Gemini function wrapper exactly like LangChain's `.invoke()`
    decision = router_llm(f'''
    You are a router for an Agentic RAG assistant.

    Route to "kb" if the user asks about:
    - Agentic RAG
    - LangGraph Agentic RAG workflow
    - retrieval grading
    - query rewriting
    - RAG architecture
    - retriever tools
    - web fallback in RAG

    Route to "direct" only for greetings, thanks, or very simple conversation.

    Question:
    {question}

    Return your response as valid JSON.
    Example:
    {{"route": "kb"}}
    ''')

    print("[Router]", decision.route)

    return {
        "current_query": question,
        "source_used": decision.route,
    }


def route_after_router(state: AgentState) -> Literal["retrieve_kb", "direct_answer"]:
    # Evaluates the outcome string to direct LangGraph execution
    if state["source_used"] == "KB":
        return "retrieve_kb"
    return "direct_answer"


# ==========================================
# 4. SANITY CHECK RUN
# ==========================================
if __name__ == "__main__":
    # Simulate a graph state payload
    mock_state: AgentState = {"question": "How does dynamic query rewriting improve an Agentic RAG system?"}
    
    # 1. Run the routing node
    updated_state = route_question(mock_state)
    print(f"Updated state after routing: {updated_state}")
    
    # 2. Check where the conditional edge routes it
    next_node = route_after_router(updated_state)
    print(f"Graph will now transition to: {next_node}")


[Router] KB
Updated state after routing: {'current_query': 'How does dynamic query rewriting improve an Agentic RAG system?', 'source_used': 'KB'}
Graph will now transition to: retrieve_kb


#### 8. Node 2 — Retrieve from the Private KB

In [15]:
def retrieve_kb(state: AgentState):
    query = state["current_query"]
    docs = retriever.invoke(query)

    print(f"[KB Retriever] Query: {query}")
    print(f"[KB Retriever] Retrieved: {len(docs)} chunks")

    return {"kb_docs": docs}

if __name__ == "__main__":
    # Simulate a graph state payload
    mock_state: AgentState = {
        "question": "How does dynamic query rewriting improve an Agentic RAG system?",
        "current_query": "How does query rewriting improve Agentic RAG?"
        }
    node_output = retrieve_kb(mock_state)
    print(f"[KB Retriever] Node Output: {node_output}")

[KB Retriever] Query: How does query rewriting improve Agentic RAG?
[KB Retriever] Retrieved: 4 chunks
[KB Retriever] Node Output: {'kb_docs': [Document(id='6b1d4b33-f1d6-466e-ae66-eb1cd4f749dc', metadata={'description': 'Build a custom retrieval agent with LangGraph that decides when to search a vector store or respond directly.', 'language': 'en', 'source': 'https://docs.langchain.com/oss/python/langgraph/agentic-rag', 'start_index': 6.0, 'title': 'Build a custom RAG agent with LangGraph - Docs by LangChain'}, page_content='Build a custom RAG agent with LangGraph - Docs by LangChain'), Document(id='4cd4bed8-b699-424e-9a3f-09f7e966d029', metadata={'description': 'Build a custom retrieval agent with LangGraph that decides when to search a vector store or respond directly.', 'language': 'en', 'source': 'https://docs.langchain.com/oss/python/langgraph/agentic-rag', 'start_index': 6.0, 'title': 'Build a custom RAG agent with LangGraph - Docs by LangChain'}, page_content='Build a custom RA

#### 9. Node 3 — Grade Private KB Evidence
###### This is the key step that decides whether we can answer from the private KB or need web fallback.

In [16]:
kb_grader_llm = gemini_factory.with_structured_output(EvidenceGrade)

def grade_kb_evidence(state: AgentState):
    question = state["question"]

    context = "\n\n".join(
        f"Source: {doc.metadata.get('source')}\n{doc.page_content}"
        for doc in state["kb_docs"]
    )

    grade = kb_grader_llm.invoke(f'''
    You are an evidence grader.

    Question:
    {question}

    Private KB evidence:
    {context}

    Can this private KB evidence answer the question?
    Return "good" if it can answer.
    Return "weak" if it cannot answer or is incomplete.

    Return your response as valid JSON.
    Example:
    {{"grade": "good"}}
    ''')

    print("[KB Grader]", grade.grade)

    return {"kb_grade": grade.grade}


def decide_after_kb_grade(state: AgentState) -> Literal["generate_from_kb", "search_web"]:
    if state["kb_grade"] == "good":
        return "generate_from_kb"
    return "search_web"


# ==========================================
# SANITY CHECK RUN
# ==========================================
# if __name__ == "__main__":
#     # Simulate a graph state payload
#     # mock_state: AgentState = {"question": "How does dynamic query rewriting improve an Agentic RAG system?"}
    
#     # 1. Run the routing node
#     updated_grade_kb_state = grade_kb_evidence(mock_state)
#     print(f"Updated state after routing: {updated_grade_kb_state}")
    
#     # 2. Check where the conditional edge routes it
#     next_node = decide_after_kb_grade(updated_grade_kb_state)
#     print(f"Graph will now transition to: {next_node}")

#### Node 4 — Tavily Web Search Fallback

###### This node runs only when the private KB evidence is weak.

###### We search the web using the original question, or the current rewritten query if a rewrite already happened.

In [17]:
def search_web(state: AgentState):
    query = state["current_query"]

    print(f"[Tavily Search] Query: {query}")

    result = web_search.invoke({"query": query})

    # Tavily can return dict/list structures depending on package version.
    # Convert it into readable text for grading and generation.
    if isinstance(result, dict):
        answer = result.get("answer", "")
        results = result.get("results", [])
        lines = []
        if answer:
            lines.append(f"Tavily answer: {answer}")

        for item in results:
            title = item.get("title", "")
            url = item.get("url", "")
            content = item.get("content", "")
            lines.append(f"Title: {title}\nURL: {url}\nContent: {content}")

        web_text = "\n\n".join(lines) if lines else str(result)
    else:
        web_text = str(result)

    print("[Tavily Search] Result characters:", len(web_text))

    return {
        "web_results": web_text,
        "source_used": "web",
    }

#### Node 5 — Grade Web Evidence

In [18]:
web_grader_llm = gemini_factory.with_structured_output(EvidenceGrade)

def grade_web_evidence(state: AgentState):
    question = state["question"]
    web_result = state["web_results"]

    grade = web_grader_llm.invoke(f'''
    You are an evidence grader.

    Question:
    {question}

    Web search results:
    {web_result}

    Can this web evidence answer the question?
    Return "good" if it can answer.
    Return "weak" if it cannot answer or is incomplete.

    Return your response as valid JSON.
    Example:
    {{"grade": "good"}}
    ''')

    print("[Web Grader]", grade.grade)
    return {"kb_grade": grade.grade}  # Reusing kb_grade for simplicity

MAX_RETRIES = 2

def decide_after_web_grade(
    state: AgentState
) -> Literal["generate_from_web", "rewrite_query", "answer_inaufficient"]:

    if state["kb_grade"] == "good":
        return "generate_from_web"
    if state["retry_count"] < MAX_RETRIES:
        return "rewrite_query"
    return "answer_inaufficient"

#### Node 6 -- Rewrite the Query
###### If both KB and Web evidence are weak, we rewrite once and then try the KB again.
###### Why back to KB first?
###### Because the rewritten query may help retrieve brtter private evidence before relying on web search again.

In [19]:
from langchain_core.messages import HumanMessage

def rewrite_query(state: AgentState):
    question = state["question"]
    retry_count = state.get("retry_count", 0) + 1
    # web_result = state["web_results"]

    # Use the Gemini LLM to rewrite the query based on the web results
    rewritten_query = llm.invoke(
        # model="gemini-3.8-flash",
        input=f'''
        Rewrite the question for better retrieval and web search.

        Rules:
        - Preserve original intent
        - Make it specific and search-friendly
        - Do not answer
        - Return only the rewritten query

        Original Question:
        {question}
        ''',
    ).content.strip()

    print("[Query Rewriter] Rewritten Query:", rewritten_query)

    return {
        "current_query": rewritten_query,
        "retry_count": state.get("retry_count", 0) + 1,
    }


#### Node 7 -- Generate from Private KB
######

In [20]:
def generate_from_kb(state: AgentState):
    question = state["question"]

    context = "\n\n".join(
        f"[KB Source: {doc.metadata.get('source')}]\n{doc.page_content}"
        for doc in state["kb_docs"]
    )

    answer = llm.invoke(
        f'''
        You are a technical instructor.
        Answer using only the private KB context.

        Rules:
        - Beginner friendly explanations.
        - Do not invent unsupported details.
        - Mention the the answer based on the private KB.
        - Include source type: Private KB.

        Question:
        {question}

        Private KB Context:
        {context}
        ''').content

    print("[Answer Generator] Generated Answer from Private KB:", answer[:200], "\n\n")

    return {
        "answer": answer,
        "source_used": "private_kb",
    }

#### Node 8 -- Generate From web Search

In [ ]:
def generate_from_web_search(state: AgentState):
    question = state["question"]
    web_context = state["web_results"]

    answer = llm.invoke(f'''
        You are a technical instructor.
        The private KB was insufficient, so web search was used.
        Answer using ONLY the web search context.

        Rules:
        - Beginner-friendly explanations.
        - Do not invent unsupported details.
        - Mention that the answer is based on Tavily web search.
        - Include source type: Web Search.
        - If URLs are present in context, include the most useful URLs.

        Question:
        {question}
        
        Web Search Context:
        {web_context}
        ''').content

    print("[Answer Generator] Generated Answer from Web Search:", answer[:200], "\n\n")


    return {
        "answer": answer,
        "source_used": "web_search",
    }